# Phutball Elo tournament (A100)

Round robin between the self-play checkpoints close enough in strength to beat each other: the switch checkpoint (it50,
the reference = 1000), the bests at it80 / it90 / it100 (= the final), the 17x13 final (played on 21x15), and the best
PBT member. 32x16 Gumbel search without root noise, 64 games per colour per pair, 360-turn cap. Results in
`My Drive/phutball/elo/elo.json`. Add your games with HUMAN = ["you:W-L-D vs it100"].

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/phutball'

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install mctx==0.0.6 2>&1 | tail -1

In [ ]:
import os, subprocess, sys
S = f'{DRIVE}/selfplay'
PLAYERS = {'it50': f'{S}/selfplay_21x15_switch_s128_it50.pkl', 'it80': f'{S}/selfplay_21x15_best_it80.pkl',
           'it90': f'{S}/selfplay_21x15_best_it90.pkl', 'it100': f'{S}/selfplay_21x15_best_it100.pkl',
           '17x13': f'{S}/selfplay_17x13_final.pkl', 'pbt': f'{DRIVE}/grpo2/best_latest.pkl'}
PLAYERS = {k: v for k, v in PLAYERS.items() if os.path.exists(v) or print('missing, skipped:', k, v)}
HUMAN = []   # e.g. ["you:3-7-0 vs it100"] after tonight's games
os.makedirs(f'{DRIVE}/elo', exist_ok=True)
args = ' '.join(f'--player {k}={v}' for k, v in PLAYERS.items()) + ' ' + ' '.join(f'--human "{h}"' for h in HUMAN)
cmd = f'{sys.executable} -u -m expert.elo_tournament {args} --anchor it50 --games 64 --sims 32 --out {DRIVE}/elo/elo.json'
p = subprocess.Popen(cmd, shell=True, cwd='/content/phutball-jax', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end='')
print('exit code', p.wait())

## Calibrate the browser opponent

The browser plays the network with PUCT at 400 playouts; the tournament uses Gumbel at 32x16. This plays PUCT-400 against
Gumbel 32/64/128 x16 (same network: the final), one process per budget in parallel (5 games per colour each), and gives
the Elo gap, which puts "final + browser search", and your games against it, on the tournament scale.

In [ ]:
import subprocess, sys, os, json
env = dict(os.environ, XLA_PYTHON_CLIENT_PREALLOCATE='false')      # several JAX processes share the GPU
procs = {}
for sims in (32, 64, 128):
    cmd = (f'{sys.executable} -u -m expert.puct_calibration --params {S}/selfplay_21x15_best_it100.pkl --playouts 400 '
           f'--gumbel-sims {sims} --games 5 --out {DRIVE}/elo/puct_vs_gumbel{sims}.json')
    procs[sims] = subprocess.Popen(cmd, shell=True, cwd='/content/phutball-jax', env=env,
                                   stdout=open(f'{DRIVE}/elo/puct_vs_gumbel{sims}.log', 'w'), stderr=subprocess.STDOUT)
for sims, p in procs.items():
    print(sims, 'exit code', p.wait())
    print(open(f'{DRIVE}/elo/puct_vs_gumbel{sims}.log').read().strip().splitlines()[-1])